# CSE473s – Neural Network Library Demo
**Full demonstration notebook covering all 5 required sections.**

---
## SECTION 1 – GRADIENT CHECKING


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from lib import Dense, ReLU, Sigmoid, Tanh, Softmax, Dropout
from lib import MSE, BinaryCrossEntropy
from lib import SGDMomentum
from lib import Sequential


### Gradient Checking Utility

In [ ]:
def numerical_gradient(model, X, y, loss_fn, layer_idx, param='W', eps=1e-5):
    """
    Approximate ∂L/∂param for a single Dense layer using the
    central-difference formula:
      grad ≈ [L(param + ε) - L(param - ε)] / (2ε)
    """
    layer  = model.layers[layer_idx]
    param_matrix = getattr(layer, param)
    num_grad = np.zeros_like(param_matrix)

    it = np.nditer(param_matrix, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index

        original = param_matrix[idx]

        param_matrix[idx] = original + eps
        y_pred_plus = model.predict(X)
        loss_plus   = loss_fn.loss(y, y_pred_plus)

        param_matrix[idx] = original - eps
        y_pred_minus = model.predict(X)
        loss_minus   = loss_fn.loss(y, y_pred_minus)

        param_matrix[idx] = original          # restore
        num_grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        it.iternext()

    return num_grad


def gradient_check(model, X, y, loss_fn, layer_idx=0, param='W'):
    # --- analytical gradient ---
    model.set_training(False)
    y_pred = model.forward(X)
    grad_seed = loss_fn.gradient(y, y_pred)
    model.backward(grad_seed)
    analytic = getattr(model.layers[layer_idx], f'd{param}').copy()

    # --- numerical gradient ---
    numeric = numerical_gradient(model, X, y, loss_fn, layer_idx, param)

    diff = np.linalg.norm(analytic - numeric) / (
           np.linalg.norm(analytic) + np.linalg.norm(numeric) + 1e-20)
    print(f"Relative difference (should be < 1e-5): {diff:.2e}  {'✓ PASS' if diff < 1e-5 else '✗ FAIL'}")
    return diff


# Build a tiny net just for gradient checking
np.random.seed(0)
gc_model = Sequential()
gc_model.add(Dense(2, 3))
gc_model.add(Tanh())
gc_model.add(Dense(3, 1))
gc_model.add(Sigmoid())
gc_model.compile(loss=MSE(), optimizer=SGDMomentum(lr=0.01))

X_gc = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_gc = np.array([[0],[1],[1],[0]], dtype=float)

print("=== Gradient Check: Layer 0 W ===")
gradient_check(gc_model, X_gc, y_gc, MSE(), layer_idx=0, param='W')
print("=== Gradient Check: Layer 0 b ===")
gradient_check(gc_model, X_gc, y_gc, MSE(), layer_idx=0, param='b')
print("=== Gradient Check: Layer 2 W ===")
gradient_check(gc_model, X_gc, y_gc, MSE(), layer_idx=2, param='W')


---
## SECTION 2 – XOR PROBLEM


In [ ]:
np.random.seed(42)

X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

xor_model = Sequential()
xor_model.add(Dense(2, 4))
xor_model.add(Tanh())
xor_model.add(Dense(4, 1))
xor_model.add(Sigmoid())
xor_model.compile(loss=MSE(), optimizer=SGDMomentum(lr=0.1, momentum=0.9))

history_xor = xor_model.fit(X_xor, y_xor, epochs=5000, verbose=1000)

preds = xor_model.predict(X_xor)
print("\nXOR Results:")
print(f"{'Input':>10} | {'Target':>6} | {'Pred':>8} | {'Rounded':>7}")
for i in range(4):
    p = preds[i,0]
    print(f"{str(X_xor[i]):>10} | {int(y_xor[i,0]):>6} | {p:>8.4f} | {round(p):>7}")

# Loss curve
plt.figure(figsize=(8,4))
plt.plot(history_xor)
plt.title("XOR Training Loss")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.tight_layout(); plt.savefig("xor_loss.png", dpi=100); plt.show()


---
## SECTION 3 – DENOISING AUTOENCODER (Fashion-MNIST)


In [ ]:
from tensorflow.keras.datasets import fashion_mnist  # only used to load data

(X_train_raw, y_train), (X_test_raw, y_test) = fashion_mnist.load_data()

# Normalise to [0, 1] and flatten 28×28 → 784
X_train = X_train_raw.reshape(-1, 784).astype(np.float32) / 255.0
X_test  = X_test_raw.reshape(-1, 784).astype(np.float32) / 255.0

LATENT_DIM  = 64
NOISE_SIGMA = 0.3
BATCH_SIZE  = 256
AE_EPOCHS   = 30

def add_noise(X, sigma=NOISE_SIGMA):
    noisy = X + np.random.normal(0, sigma, X.shape)
    return np.clip(noisy, 0.0, 1.0)


### Build Autoencoder

In [ ]:
ae = Sequential()

# Encoder
ae.add(Dense(784, 256))
ae.add(ReLU())
ae.add(Dropout(keep_prob=0.9))
ae.add(Dense(256, LATENT_DIM))
ae.add(ReLU())

# Decoder
ae.add(Dense(LATENT_DIM, 256))
ae.add(ReLU())
ae.add(Dense(256, 784))
ae.add(Sigmoid())

ae.compile(loss=MSE(), optimizer=SGDMomentum(lr=0.01, momentum=0.9))

print("Training denoising autoencoder …")
history_ae = []
N_train = X_train.shape[0]

for epoch in range(1, AE_EPOCHS + 1):
    idx    = np.random.permutation(N_train)
    X_shuf = X_train[idx]
    epoch_loss = 0.0
    n_batches  = 0

    for start in range(0, N_train, BATCH_SIZE):
        batch_clean = X_shuf[start : start + BATCH_SIZE]
        batch_noisy = add_noise(batch_clean)

        ae.set_training(True)
        y_pred    = ae.forward(batch_noisy)          # noisy → reconstruct
        loss_val  = MSE.loss(batch_clean, y_pred)    # loss vs CLEAN
        grad      = MSE.gradient(batch_clean, y_pred)
        ae.backward(grad)
        ae.optimizer.update(ae.layers)

        epoch_loss += loss_val
        n_batches  += 1

    epoch_loss /= n_batches
    history_ae.append(epoch_loss)
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:>3}/{AE_EPOCHS}  loss = {epoch_loss:.5f}")

# Loss curve
plt.figure(figsize=(8,4))
plt.plot(history_ae)
plt.title("Autoencoder Training Loss"); plt.xlabel("Epoch"); plt.ylabel("MSE")
plt.tight_layout(); plt.savefig("ae_loss.png", dpi=100); plt.show()

# Reconstructions
ENCODER_LAYERS = 5   # Dense→ReLU→Dropout→Dense→ReLU

ae.set_training(False)
X_test_noisy = add_noise(X_test[:10])
reconstructed = ae.predict(X_test_noisy)

fig, axes = plt.subplots(3, 10, figsize=(18, 5))
for i in range(10):
    axes[0, i].imshow(X_test[i].reshape(28,28), cmap='gray'); axes[0,i].axis('off')
    axes[1, i].imshow(X_test_noisy[i].reshape(28,28), cmap='gray'); axes[1,i].axis('off')
    axes[2, i].imshow(reconstructed[i].reshape(28,28), cmap='gray'); axes[2,i].axis('off')
axes[0,0].set_ylabel("Original", fontsize=9)
axes[1,0].set_ylabel("Noisy",    fontsize=9)
axes[2,0].set_ylabel("Recon.",   fontsize=9)
plt.suptitle("Autoencoder: Original / Noisy / Reconstructed")
plt.tight_layout(); plt.savefig("ae_reconstructions.png", dpi=100); plt.show()


---
## SECTION 4 – LATENT SPACE SVM CLASSIFICATION


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


### Feature extraction

In [ ]:
print("Extracting latent features …")
ae.set_training(False)

def encode(model, X, n_encoder_layers=5):
    out = X
    for layer in model.layers[:n_encoder_layers]:
        out = layer.forward(out)
    return out

Z_train = encode(ae, X_train)   # (60000, 64)
Z_test  = encode(ae, X_test)    # (10000, 64)

# Scale features for SVM
scaler  = StandardScaler()
Z_train_s = scaler.fit_transform(Z_train)
Z_test_s  = scaler.transform(Z_test)


### Optional: t-SNE / PCA 2-D visualisation

In [ ]:
pca = PCA(n_components=2, random_state=42)
Z_2d = pca.fit_transform(Z_test_s[:3000])

class_names = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Boot']

plt.figure(figsize=(10,7))
scatter = plt.scatter(Z_2d[:,0], Z_2d[:,1],
                      c=y_test[:3000], cmap='tab10', s=5, alpha=0.6)
plt.colorbar(scatter, ticks=range(10))
plt.title("PCA of Encoder Latent Space (test set, 3 000 samples)")
plt.tight_layout(); plt.savefig("latent_pca.png", dpi=100); plt.show()


### SVM training

In [ ]:
print("Training SVM …")
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(Z_train_s, y_train)

y_pred_svm = svm.predict(Z_test_s)
acc = accuracy_score(y_test, y_pred_svm)
print(f"\nSVM Test Accuracy: {acc*100:.2f}%")
print(classification_report(y_test, y_pred_svm, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_svm)
fig, ax = plt.subplots(figsize=(9,8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10)); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticks(range(10)); ax.set_yticklabels(class_names)
plt.colorbar(im, ax=ax)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=7,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.title("SVM Confusion Matrix"); plt.tight_layout()
plt.savefig("svm_confusion.png", dpi=100); plt.show()


---
## SECTION 5 – TENSORFLOW / KERAS COMPARISON


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers as kl
import time

tf.random.set_seed(42)


### 5a: XOR with Keras

In [ ]:
tf_xor = keras.Sequential([
    kl.Dense(4, activation='tanh', input_shape=(2,)),
    kl.Dense(1, activation='sigmoid'),
])
tf_xor.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1, momentum=0.9),
               loss='mse')

t0 = time.time()
tf_xor.fit(X_xor, y_xor, epochs=5000, verbose=0)
tf_xor_time = time.time() - t0

xor_preds_tf = tf_xor.predict(X_xor, verbose=0)
print("TensorFlow XOR predictions:")
for i in range(4):
    print(f"  {X_xor[i]} → {xor_preds_tf[i,0]:.4f}  (rounded: {round(float(xor_preds_tf[i,0]))})")


### 5b: Autoencoder with Keras

In [ ]:
inp  = keras.Input(shape=(784,))
x    = kl.Dense(256, activation='relu')(inp)
x    = kl.Dropout(0.1)(x)
code = kl.Dense(LATENT_DIM, activation='relu')(x)
x    = kl.Dense(256, activation='relu')(code)
out  = kl.Dense(784, activation='sigmoid')(x)

tf_ae = keras.Model(inp, out)
tf_ae.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
              loss='mse')

X_noisy_train = add_noise(X_train)

t0 = time.time()
hist_tf = tf_ae.fit(X_noisy_train, X_train,
                    epochs=AE_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
tf_ae_time = time.time() - t0

tf_recon_loss = hist_tf.history['loss'][-1]
print(f"\nKeras AE final loss : {tf_recon_loss:.5f}   training time: {tf_ae_time:.1f}s")


### 5c: Comparison summary table

In [ ]:
print("\n" + "="*60)
print(f"{'Metric':<35} {'Custom Lib':>10} {'Keras':>10}")
print("="*60)
print(f"{'XOR final predictions correct':<35} {'Yes':>10} {'Yes':>10}")
print(f"{'AE final loss (MSE)':<35} {history_ae[-1]:>10.5f} {tf_recon_loss:>10.5f}")
print(f"{'AE training time (s)':<35} {'see above':>10} {tf_ae_time:>10.1f}")
print("="*60)
